# Packages

In [ ]:
# Load libraries
library(DESeq2)
library(tximport)
library(org.Mm.eg.db) # For gene ID to symbol mapping
library(AnnotationDbi)
library(Seurat)
library(SeuratDisk)
library(clusterProfiler)
library(dplyr)
library(tidyverse) 

library(ggplot2)
library(pheatmap)
library(grid)
library(DOSE)
library(ggrepel)

library(zellkonverter) # Alternative for reading .h5ad as SingleCellExperiment
library(reticulate)

library(Matrix)

library(CellChat)
library(patchwork)  # for combining plots

library(fgsea)
library(msigdbr)
library(reshape2) # Load reshape2 package for melt

# Bulk RNA-Seq Differential Expression for Muscle

##  TxImport (Prepare data)

In [ ]:
# 1. Read the count data (GSE268971_Muscle_counts_file.txt.gz)
muscle_counts <- read.table("/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/1st_attempt_count_table_data/GSE268971_Muscle_counts_file.txt.gz",
                            header = TRUE, sep = "\t", row.names = 1)

# 2. Load muscle sample metadata (with detailed columns)
sample_sheet_muscle <- read.csv("/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/1st_attempt_count_table_data/muscle_sample_metadata_filtered.csv",
                               header = TRUE, sep = ",")

# 3. Align sample names to count matrix columns:
# Your count matrix columns have sample names like "IG13Aligned.out.bam"
# So create a matching column in sample sheet to match full names in count matrix
sample_sheet_muscle$sample_name_full <- paste0(sample_sheet_muscle$sample_name, "Aligned.out.bam")

# Set the rownames of sample sheet to these full names for indexing
rownames(sample_sheet_muscle) <- sample_sheet_muscle$sample_name_full

# 4. Now subset muscle_counts by columns (samples) present in sample sheet
muscle_counts <- muscle_counts[, rownames(sample_sheet_muscle)]

## Create DESeq()

In [ ]:
# 5. Build DESeq object and normalize
dds_muscle <- DESeqDataSetFromMatrix(
  countData = muscle_counts,
  colData = sample_sheet_muscle,
  design = ~ condition
)
dds_muscle <- estimateSizeFactors(dds_muscle)

# 6. Filter low counts and run DESeq
keep <- rowSums(counts(dds_muscle)) >= 10
dds_muscle <- dds_muscle[keep,]
# Fit the model
dds_muscle <- DESeq(dds_muscle)

# normalized dataset object
ntd <- normTransform(dds_muscle)

## Calculate DEGs

In [ ]:
# 4. Extract DEG results: aging effect and intervention effects
res_age_raw  <- results(dds_muscle, contrast = c("condition", "age", "ctrl"))
res_intv_raw  <- results(dds_muscle, contrast = c("condition", "combi", "age"))
res_intv1_raw  <- results(dds_muscle, contrast=c('condition', 'DR', 'age'))
res_intv2_raw  <- results(dds_muscle, contrast=c('condition', 'sAct', 'age'))
res_combi_ctrl_raw  <- results(dds_muscle, contrast = c("condition", "combi", "ctrl"))

# Add LFC shrinkage
res_age  <- lfcShrink(dds_muscle, contrast = c("condition", "age", "ctrl"),
  res = res_age_raw,
  type = "ashr"
)

res_intv <- lfcShrink(
  dds_muscle,
  contrast = c("condition", "combi", "age"),
  res = res_intv_raw,
  type = "ashr"
)

res_intv1 <- lfcShrink(
  dds_muscle,
  contrast = c("condition", "DR", "age"),
  res = res_intv1_raw,
  type = "ashr"
)

res_intv2 <- lfcShrink(
  dds_muscle,
  contrast = c("condition", "sAct", "age"),
  res = res_intv2_raw,
  type = "ashr"
)

res_combi_ctrl <- lfcShrink(
  dds_muscle,
  contrast = c("condition", "combi", "ctrl"),
  res = res_combi_ctrl_raw,
  type = "ashr"
)

In [ ]:
plotMA(res_age, main = "MA Plot: Age vs Ctrl (Muscle)", ylim = c(-5, 5))

In [ ]:
# Convert to dataframes
res_age_df <- as.data.frame(res_age)
res_intv_df <- as.data.frame(res_intv)
res_combi_ctrl_df <- as.data.frame(res_combi_ctrl)
res_intv1_df <- as.data.frame(res_intv1)
res_intv2_df <- as.data.frame(res_intv2)
# Clean Ensembl IDs (remove version suffixes) in all
clean_ensembl_ids_age <- sub("\\.[0-9]+$", "", rownames(res_age_df))
clean_ensembl_ids_intv <- sub("\\.[0-9]+$", "", rownames(res_intv_df))
clean_ensembl_ids_combi_ctrl <- sub("\\.[0-9]+$", "", rownames(res_combi_ctrl_df))

# Replace rownames with cleaned IDs (IMPORTANT)
rownames(res_age_df) <- clean_ensembl_ids_age
rownames(res_intv_df) <- clean_ensembl_ids_intv
rownames(res_combi_ctrl_df) <- clean_ensembl_ids_combi_ctrl
rownames(res_intv1_df) <- sub("\\.[0-9]+$", "", rownames(res_intv1_df))
rownames(res_intv2_df) <- sub("\\.[0-9]+$", "", rownames(res_intv2_df))

# 5. Filter genes significantly altered by aging
deg_age <- subset(res_age_df, padj < 0.05 & abs(log2FoldChange) > 0)

# 6. Identify rescued genes reversed by combined intervention
rescued_genes <- deg_age[which(sign(deg_age$log2FoldChange) != sign(res_intv_df[rownames(deg_age), "log2FoldChange"])), ]

# Add combined intervention vs control results to rescued genes
rescued_genes$padj_combi_ctrl <- res_combi_ctrl_df[rownames(rescued_genes), "padj"]
rescued_genes$log2FC_combi_ctrl <- res_combi_ctrl_df[rownames(rescued_genes), "log2FoldChange"]

# 7. Filter rescued genes that go back close to normal
rescued_to_normal <- rescued_genes[abs(rescued_genes$log2FC_combi_ctrl) < 0.5, ]

In [ ]:
#Calculate DEGs not rescued by any intervention
rescued_combi <- rownames(deg_age)[sign(deg_age$log2FoldChange) != sign(res_intv_df[rownames(deg_age), 'log2FoldChange'])]
rescued_DR    <- rownames(deg_age)[sign(deg_age$log2FoldChange) != sign(res_intv1_df[rownames(deg_age), 'log2FoldChange'])]
rescued_sAct  <- rownames(deg_age)[sign(deg_age$log2FoldChange) != sign(res_intv2_df[rownames(deg_age), 'log2FoldChange'])]
rescued_all   <- unique(c(rescued_combi, rescued_DR, rescued_sAct))
non_rescued_genes <- setdiff(rownames(deg_age), rescued_all)
non_rescued_df <- deg_age[non_rescued_genes, ]
head(non_rescued_df)

In [ ]:
# Quick diagnostics:
cat("rescued_combi:", length(rescued_combi), "\n")
cat("rescued_DR:", length(rescued_DR), "\n")
cat("rescued_sAct:", length(rescued_sAct), "\n")
cat("rescued_all:", length(rescued_all), "\n")
cat("non_rescued:", nrow(non_rescued_df), "\n")

In [ ]:
# 8. Map Ensembl IDs to gene symbols (already cleaned rownames)
add_symbols <- function(df) {
  gene_symbols <- mapIds(org.Mm.eg.db,
                         keys = rownames(df),
                         column = "SYMBOL",
                         keytype = "ENSEMBL",
                         multiVals = "first")
  gene_symbols_filled <- gene_symbols
  na_idx <- is.na(gene_symbols_filled)
  gene_symbols_filled[na_idx] <- names(gene_symbols)[na_idx]
  df$Symbol <- gene_symbols_filled
  return(df)
}

rescued_genes <- add_symbols(rescued_genes)
rescued_to_normal <- add_symbols(rescued_to_normal)
deg_age <- add_symbols(deg_age)
non_rescued_df <- add_symbols(non_rescued_df)

# Intervention Comparison for Muscle

In [ ]:
# 9. Evaluate restoration by single interventions DR and sAct

# Extract DESeq2 results for interventions
res_DR <- results(dds_muscle, contrast = c("condition", "DR", "age"))
res_sAct <- results(dds_muscle, contrast = c("condition", "sAct", "age"))

# Convert these to data frames and clean Ensembl IDs as before
res_DR_df <- as.data.frame(res_DR)
res_sAct_df <- as.data.frame(res_sAct)

clean_ids_DR <- sub("\\.[0-9]+$", "", rownames(res_DR_df))
clean_ids_sAct <- sub("\\.[0-9]+$", "", rownames(res_sAct_df))
rownames(res_DR_df) <- clean_ids_DR
rownames(res_sAct_df) <- clean_ids_sAct

# Reversal gene list (already cleaned)
reversal_genes <- rownames(rescued_to_normal)

# Assemble fold changes in one data frame for reversal genes
effect_comparison <- data.frame(
  Gene = reversal_genes,
  LFC_Age = res_age_df[reversal_genes, "log2FoldChange"],
  LFC_Combined = res_intv_df[reversal_genes, "log2FoldChange"],
  LFC_DR = res_DR_df[reversal_genes, "log2FoldChange"],
  LFC_sAct = res_sAct_df[reversal_genes, "log2FoldChange"]
)
print(head(effect_comparison))
# Map gene symbols from rescued_to_normal (already mapped)
symbol_lookup <- rescued_to_normal$Symbol
names(symbol_lookup) <- rownames(rescued_to_normal)
effect_comparison$Symbol <- symbol_lookup[effect_comparison$Gene]

# Order columns
effect_comparison <- effect_comparison[, c("Gene", "Symbol", setdiff(names(effect_comparison), c("Gene", "Symbol"))) ]

# Calculate restoration ratios only if sign reverses vs aging
effect_comparison$Restoration_DR <- ifelse(sign(effect_comparison$LFC_DR) != sign(effect_comparison$LFC_Age),
                                           abs(effect_comparison$LFC_DR / effect_comparison$LFC_Age), 0)
effect_comparison$Restoration_sAct <- ifelse(sign(effect_comparison$LFC_sAct) != sign(effect_comparison$LFC_Age),
                                             abs(effect_comparison$LFC_sAct / effect_comparison$LFC_Age), 0)
effect_comparison$Restoration_Combined <- ifelse(sign(effect_comparison$LFC_Combined) != sign(effect_comparison$LFC_Age),
                                                 abs(effect_comparison$LFC_Combined / effect_comparison$LFC_Age), 0)

# Summary restoration means
summary_stats <- data.frame(
  Mean_Restoration_DR = mean(effect_comparison$Restoration_DR, na.rm = TRUE),
  Mean_Restoration_sAct = mean(effect_comparison$Restoration_sAct, na.rm = TRUE),
  Mean_Restoration_Combined = mean(effect_comparison$Restoration_Combined, na.rm = TRUE)
)

summary_stats

In [ ]:
# add a column classifying whether the intervention impact on each gene is a “rescue” or “worsening” effect compared to aging
effect_comparison$Intervention_DR_Effect <- ifelse(
  sign(effect_comparison$LFC_DR) != sign(effect_comparison$LFC_Age),
  "Rescue",
  "Worsening"
)

effect_comparison$Intervention_sAct_Effect <- ifelse(
  sign(effect_comparison$LFC_sAct) != sign(effect_comparison$LFC_Age),
  "Rescue",
  "Worsening"
)

effect_comparison$Intervention_Combined_Effect <- ifelse(
  sign(effect_comparison$LFC_Combined) != sign(effect_comparison$LFC_Age),
  "Rescue",
  "Worsening"
)

head(effect_comparison)

## Split genes into those exclusively rescued by DR XOR sAct 

In [ ]:
#Split genes into those exclusively rescued by DR XOR sAct (restoration > 0 in only one)
exclusive_DR <- effect_comparison %>%
  filter((Restoration_DR > 0 & Restoration_sAct == 0))

exclusive_sAct <- effect_comparison %>%
  filter((Restoration_sAct > 0 & Restoration_DR == 0))

only_rescued_in_combi <- effect_comparison %>%
  filter((Restoration_sAct == 0 & Restoration_DR == 0))

head(exclusive_DR)
head(exclusive_sAct)
head(only_rescued_in_combi)

In [ ]:
length(effect_comparison$Gene)
length(rescued_to_normal$Symbol)
length(exclusive_DR$Gene)
length(exclusive_sAct$Gene)
length(only_rescued_in_combi$Gene)

length(deg_age$Symbol)

In [ ]:
deg_age

### Plots

In [ ]:
# Prepare long format for restoration ratios
effect_long <- effect_comparison %>%
  pivot_longer(cols = c("Restoration_DR", "Restoration_sAct", "Restoration_Combined"),
               names_to = "Intervention", values_to = "Restoration")

# Optionally set factor levels of Intervention for consistent ordering and color
effect_long$Intervention <- factor(effect_long$Intervention,
                                   levels = c("Restoration_DR", "Restoration_sAct", "Restoration_Combined"),
                                   labels = c("DR", "sAct", "Combined"))

# Plot grouped boxplot with ggplot2
ggplot(effect_long, aes(x = Intervention, y = Restoration, fill = Intervention)) +
  geom_boxplot() +
  scale_fill_manual(values = c("DR" = "#F8766D", "sAct" = "#00BFC4", "Combined" = "#7CAE00")) +
  theme_classic() +
  ylab("Restoration Ratio (vs aging)") +
  ggtitle("Comparative Restoration Efficiency of Interventions") +
  theme(legend.position = "none")

# Top10 genes most restored by combined intervention
top10_combined <- effect_comparison %>%
  arrange(desc(abs(Restoration_Combined))) %>%
  slice_head(n = 10) %>%
  rename(Restoration = Restoration_Combined) %>%
  mutate(Intervention = "Combined")

# Reshape top10 for plotting fold changes
top10_long <- top10_combined %>%
  pivot_longer(cols = c("LFC_Age", "LFC_DR", "LFC_sAct", "LFC_Combined"),
               names_to = "Condition", values_to = "Log2FoldChange") %>%
  mutate(Condition = recode(Condition,
                            LFC_Age = "Aging",
                            LFC_DR = "DR",
                            LFC_sAct = "sAct",
                            LFC_Combined = "Combined"))

# Plot barplot with genes on y axis and fold changes for conditions in groups
ggplot(top10_long, aes(x = reorder(Symbol, abs(Log2FoldChange)), y = Log2FoldChange, fill = Condition)) +
  geom_col(position = position_dodge(width = 0.8)) + # dodge bars side by side
  coord_flip() +
  ylab("Log2 Fold Change") +
  xlab("Gene") +
  ggtitle("Top 10 Genes Restored by Combined Intervention & Impact of Single Interventions") +
  scale_fill_brewer(palette = "Set1") +
  theme_classic() +
  theme(axis.text.y = element_text(size = 12, face = "bold"),
        plot.title = element_text(hjust = 0.4),
        plot.margin = margin(t = 15, r = 15, b = 5, l = 5))


In [ ]:
# Step 6 (new): Compare proportions of gene categories
category_counts <- data.frame(
  Category = c("Exclusive_DR", "Exclusive_sAct", "only_rescued_in_combi", "Other"),
  Count = c(
    nrow(exclusive_DR),
    nrow(exclusive_sAct),
    nrow(only_rescued_in_combi),
    nrow(effect_comparison) - nrow(exclusive_DR) - nrow(exclusive_sAct) - nrow(only_rescued_in_combi)
  )
)

category_counts <- category_counts %>%
  mutate(Proportion = Count / sum(Count))

# Barplot of proportions
ggplot(category_counts, aes(x = Category, y = Proportion, fill = Category)) +
  geom_bar(stat = "identity") +
  geom_text(aes(label = scales::percent(Proportion, accuracy = 0.1)),
            vjust = -0.5, size = 3.5) +
  scale_fill_manual(values = c("Exclusive_DR" = "#F8766D",
                               "Exclusive_sAct" = "#00BFC4",
                               "only_rescued_in_combi" = "#7CAE00",
                               "Other" = "grey70")) +
  theme_classic() +
  ylab("Proportion of Genes") +
  ggtitle("Proportion of Genes Exclusively Rescued by Each Intervention Set") +
  theme(axis.text.x = element_text(angle = 90, vjust = 1, hjust = 1))

# Optional: Save table for downstream pathway analyses
write.csv(category_counts, "/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/results_tables/muscle_gene_category_proportions.csv", row.names = FALSE)

In [ ]:
# Convert proportions to percentage labels for the pie chart
category_counts <- category_counts %>%
  mutate(Percentage = scales::percent(Proportion, accuracy = 0.1))

# Pie chart with ggplot2
ggplot(category_counts, aes(x = "", y = Proportion, fill = Category)) +
  geom_col(width = 1, color = "white") +
  coord_polar(theta = "y") +
  geom_text(aes(label = Percentage), position = position_stack(vjust = 0.5)) +
  scale_fill_manual(values = c("Exclusive_DR" = "#F8766D",
                               "Exclusive_sAct" = "#00BFC4",
                               "only_rescued_in_combi" = "#7CAE00",
                               "Other" = "grey70")) +
  theme_void() +
  ggtitle("Proportion of Genes Exclusively Rescued by Each Intervention Set") +
  theme(plot.title = element_text(hjust = 0))

### Write out effect_comparison to csv

In [ ]:
# 12. Write effect comparison table for downstream analysis
write.csv(effect_comparison, file = "/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/results_tables/intervention_impact/muscle_intervention_impact_comparison.csv", row.names = TRUE)

#write head exclusive_DR   & exclusive_sAct
write.csv(exclusive_DR, file = "/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/results_tables/intervention_impact/muscle_intervention_impact_comparison_DR.csv", row.names = TRUE)
write.csv(exclusive_sAct, file = "/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/results_tables/intervention_impact/muscle_intervention_impact_comparison_sAct.csv", row.names = TRUE)
# write out only rescued in combi genes
write.csv(only_rescued_in_combi, file = "/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/results_tables/intervention_impact/muscle_intervention_impact_comparison_only_rescued_in_combi.csv", row.names = TRUE)

# Check LFC distribution

In [ ]:
# Compare LFC distributions between pipelines
cat("\n=== LFC distribution for rescued-to-normal genes ===\n")
cat("DR LFC:   median =", median(rescued_to_normal$LFC_DR, na.rm=TRUE),
    " mean =", mean(rescued_to_normal$LFC_DR, na.rm=TRUE), "\n")
cat("sAct LFC: median =", median(rescued_to_normal$LFC_sAct, na.rm=TRUE),
    " mean =", mean(rescued_to_normal$LFC_sAct, na.rm=TRUE), "\n")
cat("Combi LFC: median =", median(rescued_to_normal$LFC_Combined, na.rm=TRUE),
    " mean =", mean(rescued_to_normal$LFC_Combined, na.rm=TRUE), "\n")

# How many DR LFCs are very close to zero?
cat("\n|DR LFC| < 0.1:", sum(abs(rescued_to_normal$LFC_DR) < 0.1, na.rm=TRUE),
    "of", nrow(rescued_to_normal), "\n")
cat("|sAct LFC| < 0.1:", sum(abs(rescued_to_normal$LFC_sAct) < 0.1, na.rm=TRUE), "\n")

# Check: are the DR/sAct effects just not flipping sign?
cat("\nDR opposes aging:", sum(sign(deg_age[rownames(rescued_to_normal), "log2FoldChange"]) !=
    sign(rescued_to_normal$LFC_DR), na.rm=TRUE), "of", nrow(rescued_to_normal), "\n")
cat("sAct opposes aging:", sum(sign(deg_age[rownames(rescued_to_normal), "log2FoldChange"]) !=
    sign(rescued_to_normal$LFC_sAct), na.rm=TRUE), "\n")

# Pathway Analysis

## GO BP

In [ ]:
# 13. GO Biological Process pathway enrichment for aging, rescued, and rescued-to-normal genes
# Cleaned Ensembl IDs for all expressed genes (background)
all_genes <- rownames(res_age_df)
# Cleaned Ensembl IDs for aging DEGs
deg_age_ids <- rownames(deg_age)
# Cleaned Ensembl IDs for rescued genes
rescued_genes_ids <- rownames(rescued_genes)
# Cleaned Ensembl IDs for rescued_to_normal genes
rescued_to_normal_ids <- rownames(rescued_to_normal)

# Run enrichGO for aging DEGs (background: all genes)
ego_age <- enrichGO(
  gene = deg_age_ids,
  universe = all_genes,
  OrgDb = org.Mm.eg.db,
  keyType = "ENSEMBL",
  ont = "BP",
  pAdjustMethod = "BH",
  pvalueCutoff = 0.05,
  qvalueCutoff = 0.2
)

# Run enrichGO for rescued genes with same universe
ego_rescued <- enrichGO(
  gene = rescued_genes_ids,
  universe = all_genes,
  OrgDb = org.Mm.eg.db,
  keyType = "ENSEMBL",
  ont = "BP",
  pAdjustMethod = "BH",
  pvalueCutoff = 0.05,
  qvalueCutoff = 0.2
)

# Run enrichGO for rescued-to-normal genes
ego_rescued_to_normal <- enrichGO(
  gene = rescued_to_normal_ids,
  universe = all_genes,
  OrgDb = org.Mm.eg.db,
  keyType = "ENSEMBL",
  ont = "BP",
  pAdjustMethod = "BH",
  pvalueCutoff = 0.05,
  qvalueCutoff = 0.2
)

ego_norm_DR <- enrichGO(gene = exclusive_DR$Gene,
                OrgDb = org.Mm.eg.db,
                keyType = 'ENSEMBL',
                ont = 'BP',
                pAdjustMethod = 'BH',
                pvalueCutoff = 0.05,
                qvalueCutoff = 0.2)

ego_norm_sAct <- enrichGO(gene = exclusive_sAct$Gene,
                OrgDb = org.Mm.eg.db,
                keyType = 'ENSEMBL',
                ont = 'BP',
                pAdjustMethod = 'BH',
                pvalueCutoff = 0.2,  # relaxed cutoff
                qvalueCutoff = 0.2)

# Cleaned Ensembl IDs for rescued_to_normal genes by combi that have no driving effect from sAct or DR alone
ego_norm_only_in_combi <- enrichGO(gene = only_rescued_in_combi$Gene,
                OrgDb = org.Mm.eg.db,
                keyType = 'ENSEMBL',
                ont = 'BP',
                pAdjustMethod = 'BH',
                pvalueCutoff = 0.2,
                qvalueCutoff = 0.2)

ego_non_rescued <- enrichGO(gene = non_rescued_genes,
                OrgDb = org.Mm.eg.db,
                keyType = 'ENSEMBL',
                ont = 'BP',
                pAdjustMethod = 'BH',
                pvalueCutoff = 0.05,
                qvalueCutoff = 0.2)

## GSEA

# Visulaize bulk Results

## Generate boxplot for top rescued gene 

In [ ]:
# 15. Generate boxplot for top rescued gene expression across conditions
# 1. Make sure rownames in 'ntd' assay are cleaned (no version suffix)
rownames_ntd <- sub("\\.[0-9]+$", "", rownames(assay(ntd)))
# Sort rescued_to_normal by padj to get rescued_to_normal_sorted
rescued_to_normal_sorted <- rescued_to_normal[order(rescued_to_normal$padj), ]

# 2. Clean rescued_to_normal_sorted rownames similarly
rownames_rescued <- sub("\\.[0-9]+$", "", rownames(rescued_to_normal_sorted))
rownames(rescued_to_normal_sorted) <- rownames_rescued

# 3. Find intersection between rescued genes and ntd genes
common_genes <- intersect(rownames_ntd, rownames_rescued)

# 4. Choose the top gene that exists in 'ntd'
top_gene_id <- common_genes[1]   # use first common gene
top_gene_symbol <- rescued_to_normal_sorted[top_gene_id, "Symbol"]

if (is.na(top_gene_symbol)) top_gene_symbol <- top_gene_id

# 5. Extract expression for the top gene based on matching cleaned rownames
expr_row_index <- which(rownames_ntd == top_gene_id)
expression_values <- assay(ntd)[expr_row_index, ]

# 6. Prepare dataframe for plotting
df_expr <- data.frame(
  Expression = expression_values,
  Condition = colData(dds_muscle)$condition
)

# 7. Plot boxplot
library(ggplot2)
ggplot(df_expr, aes(x = Condition, y = Expression, fill = Condition)) +
  geom_boxplot(outlier.shape = NA) +
  geom_jitter(width = 0.2, size = 1, alpha = 0.6) +
  labs(title = paste("Expression of", top_gene_symbol),
       y = "Normalized counts (log scale)") +
  theme_classic()


## PCA and MA-plot

In [ ]:
# 16. PCA and MA plots to visualize variability and DE patterns

# Variance stabilizing transformation, with blind=FALSE to use design info
vsd_muscle <- vst(dds_muscle, blind = FALSE)

# PCA plot by condition using DESeq2 helper plotPCA, which takes 'intgroup' argument for grouping
plotPCA(vsd_muscle, intgroup = "condition") + ggtitle("PCA by Condition - Muscle")

# MA plots for key contrasts
plotMA(res_age, main = "MA-plot: Aging vs Control - Muscle", ylim = c(-5, 5))
plotMA(res_intv, main = "MA-plot: Combined vs Aging - Muscle", ylim = c(-5, 5))

## Vulcano plots

### Ctrl vs Age

In [ ]:
# Convert res_age to dataframe
res_age_df <- as.data.frame(res_age)

# Remove version suffix from Ensembl IDs before mapping
clean_ids <- sub("\\.[0-9]+$", "", rownames(res_age_df))

# Map to gene symbols
gene_symbols <- mapIds(org.Mm.eg.db,
                       keys = clean_ids,
                       column = "SYMBOL",
                       keytype = "ENSEMBL",
                       multiVals = "first")

res_age_df$Symbol <- gene_symbols

# Fallback to Ensembl IDs if symbol not found
res_age_df$Symbol[is.na(res_age_df$Symbol)] <- rownames(res_age_df)[is.na(res_age_df$Symbol)]

# Define significance categories
res_age_df$significance <- "Not Significant"
res_age_df$significance[res_age_df$padj < 0.05 & res_age_df$log2FoldChange > 0] <- "Upregulated"
res_age_df$significance[res_age_df$padj < 0.05 & res_age_df$log2FoldChange < 0] <- "Downregulated"

# Select top genes by significance for labeling
top_genes <- res_age_df %>%
  filter(significance != "Not Significant") %>%
  arrange(padj) %>%
  slice_head(n = 15) %>%
  pull(Symbol)

# Volcano plot with ggplot2 & ggrepel
ggplot(res_age_df, aes(x = log2FoldChange, y = -log10(padj), color = significance)) +
  geom_point(alpha = 0.6) +
  scale_color_manual(values = c("Upregulated" = "red", "Downregulated" = "blue", "Not Significant" = "grey")) +
  theme_classic() +
  labs(title = "Volcano plot of Ctrl vs Aging DEGs") +
  geom_text_repel(data = subset(res_age_df, Symbol %in% top_genes),
                  aes(label = Symbol), size = 3, max.overlaps = 10)

### Age vs Combi

In [ ]:
# Convert res_intv to dataframe
res_intv_df <- as.data.frame(res_intv)

# Remove version suffix from Ensembl IDs before mapping
clean_ids <- sub("\\.[0-9]+$", "", rownames(res_intv_df))

# Map to gene symbols
gene_symbols <- mapIds(org.Mm.eg.db,
                       keys = clean_ids,
                       column = "SYMBOL",
                       keytype = "ENSEMBL",
                       multiVals = "first")

res_intv_df$Symbol <- gene_symbols

# Fallback to Ensembl IDs if symbol not found
res_intv_df$Symbol[is.na(res_intv_df$Symbol)] <- rownames(res_intv_df)[is.na(res_intv_df$Symbol)]

# Define significance categories
res_intv_df$significance <- "Not Significant"
res_intv_df$significance[res_intv_df$padj < 0.05 & res_intv_df$log2FoldChange > 0] <- "Upregulated"
res_intv_df$significance[res_intv_df$padj < 0.05 & res_intv_df$log2FoldChange < 0] <- "Downregulated"

# Select top genes by significance for labeling
top_genes <- res_intv_df %>%
  filter(significance != "Not Significant") %>%
  arrange(padj) %>%
  slice_head(n = 15) %>%
  pull(Symbol)

# Volcano plot with ggplot2 & ggrepel
ggplot(res_intv_df, aes(x = log2FoldChange, y = -log10(padj), color = significance)) +
  geom_point(alpha = 0.6) +
  scale_color_manual(values = c("Upregulated" = "red", "Downregulated" = "blue", "Not Significant" = "grey")) +
  theme_classic() +
  labs(title = "Volcano plot of Ctrl vs Aging DEGs") +
  geom_text_repel(data = subset(res_intv_df, Symbol %in% top_genes),
                  aes(label = Symbol), size = 3, max.overlaps = 10)


### Rescued by Combi

In [ ]:
# Convert muscle aging DESeq2 results to dataframe
res_age_df <- as.data.frame(res_age)

# Remove version suffix from Ensembl IDs for all rownames
clean_ids <- sub("\\.[0-9]+$", "", rownames(res_age_df))
rownames(res_age_df) <- clean_ids

# Map cleaned Ensembl IDs to gene symbols
gene_symbols <- mapIds(org.Mm.eg.db,
                       keys = clean_ids,
                       column = "SYMBOL",
                       keytype = "ENSEMBL",
                       multiVals = "first")

res_age_df$Symbol <- gene_symbols

# Use Ensembl ID if no symbol found
res_age_df$Symbol[is.na(res_age_df$Symbol)] <- rownames(res_age_df)[is.na(res_age_df$Symbol)]

# Define significance categories
res_age_df$significance <- "Not Significant"
res_age_df$significance[res_age_df$padj < 0.05 & res_age_df$log2FoldChange > 0] <- "Upregulated"
res_age_df$significance[res_age_df$padj < 0.05 & res_age_df$log2FoldChange < 0] <- "Downregulated"

# Ensure rescued_genes rownames are also cleaned (remove versions)
#rownames(rescued_genes) <- sub("\\.[0-9]+$", "", rownames(rescued_genes))

# Mark rescued genes in significance category
rescued_genes_ids <- rownames(rescued_genes)  # your rescued genes object with clean IDs
res_age_df$highlight <- ifelse(rownames(res_age_df) %in% rescued_genes_ids & 
                               res_age_df$significance != "Not Significant",
                               "Rescued", "Other")

# Set factor levels to control color mapping
res_age_df$highlight <- factor(res_age_df$highlight, levels = c("Other", "Rescued"))
colors <- c("Other" = "grey", "Rescued" = "purple")

# Select top rescued genes by padj for labeling
top_rescued_genes <- res_age_df %>%
  filter(highlight == "Rescued") %>%
  arrange(padj) %>%
  slice_head(n = 15) %>%
  pull(Symbol)

# Plot volcano plot with rescued genes highlighted and labeled
ggplot(res_age_df, aes(x = log2FoldChange, y = -log10(padj), color = highlight)) +
  geom_point(alpha = 0.7) +
  scale_color_manual(values = colors) +
  theme_classic() +
  labs(title = "Volcano plot with rescued genes highlighted - Muscle",
       x = "log2 Fold Change",
       y = "-log10 Adjusted p-value") +
  geom_text_repel(data = subset(res_age_df, Symbol %in% top_rescued_genes),
                  aes(label = Symbol),
                  size = 3,
                  max.overlaps = 10)

#### Rescued by combi driven by SAct exclusively

In [ ]:
# Convert DESeqResults to data frame and add gene symbols
res_age_df <- as.data.frame(res_age)
# Remove version numbers (e.g., '.5', '.12') from Ensembl IDs
rownames(res_age_df) <- sub("\\..*$", "", rownames(res_age_df))

# Map cleaned Ensembl IDs to Gene Symbols
res_age_df$Symbol <- mapIds(org.Mm.eg.db,
                            keys = rownames(res_age_df),
                            column = "SYMBOL",
                            keytype = "ENSEMBL",
                            multiVals = "first")

# If no match is found, use Ensembl ID as fallback
res_age_df$Symbol[is.na(res_age_df$Symbol)] <- rownames(res_age_df)[is.na(res_age_df$Symbol)]

# Define significant regulated genes (padj < 0.05 & |log2FC| > 1)
res_age_df$significant <- ifelse(res_age_df$padj < 0.05 & abs(res_age_df$log2FoldChange) > 0, "Significant", "Not Significant")

# Mark rescued genes among significant ones
res_age_df$highlight <- ifelse(rownames(res_age_df) %in% exclusive_sAct$Gene & res_age_df$significant == "Significant",
                               "Rescued", "Other")

# Colors: rescued purple, others grey
colors <- c("Rescued" = "purple", "Other" = "grey")

# Select rescued genes for labeling (top 15 by padj)
top_rescued_genes <- res_age_df %>%
  filter(highlight == "Rescued") %>%
  arrange(padj) %>%
  slice_head(n = 15) %>%
  pull(Symbol)

ggplot(res_age_df, aes(x = log2FoldChange, y = -log10(padj), color = highlight)) +
  geom_point(alpha = 0.7) +
  scale_color_manual(values = colors) +
  theme_classic() +
  labs(title = "Volcano plot with rescued genes driven by sAct highlighted") +
  geom_text_repel(data = subset(res_age_df, Symbol %in% top_rescued_genes),
                  aes(label = Symbol), size = 3, max.overlaps = 10)

#### Rescued by combi driven by DR exclusively

In [ ]:
# Convert DESeqResults to data frame
res_age_df <- as.data.frame(res_age)

# Remove version numbers (e.g., '.5', '.12') from Ensembl IDs
rownames(res_age_df) <- sub("\\..*$", "", rownames(res_age_df))

# Map Ensembl IDs (without version suffixes) to Gene Symbols
res_age_df$Symbol <- mapIds(org.Mm.eg.db,
                            keys = rownames(res_age_df),
                            column = "SYMBOL",
                            keytype = "ENSEMBL",
                            multiVals = "first")

# Use Ensembl IDs as fallback for unmatched symbols
res_age_df$Symbol[is.na(res_age_df$Symbol)] <- rownames(res_age_df)[is.na(res_age_df$Symbol)]

# Define significantly regulated genes (padj < 0.05 & |log2FC| > 1)
res_age_df$significant <- ifelse(res_age_df$padj < 0.05 & abs(res_age_df$log2FoldChange) > 1,
                                 "Significant", "Not Significant")

# Mark rescued genes among significant ones
res_age_df$highlight <- ifelse(rownames(res_age_df) %in% exclusive_DR$Gene &
                               res_age_df$significant == "Significant",
                               "Rescued", "Other")

# Colors: rescued purple, others grey
colors <- c("Rescued" = "purple", "Other" = "grey")

# Select rescued genes for labeling (top 15 by padj)
top_rescued_genes <- res_age_df %>%
  filter(highlight == "Rescued") %>%
  arrange(padj) %>%
  slice_head(n = 15) %>%
  pull(Symbol)

# Volcano plot
ggplot(res_age_df, aes(x = log2FoldChange, y = -log10(padj), color = highlight)) +
  geom_point(alpha = 0.7) +
  scale_color_manual(values = colors) +
  theme_classic() +
  labs(title = "Volcano plot with rescued genes driven by DR highlighted") +
  geom_text_repel(data = subset(res_age_df, Symbol %in% top_rescued_genes),
                  aes(label = Symbol), size = 3, max.overlaps = 10)

## Heatmaps

In [ ]:
## Aging

In [ ]:
# Generate heatmap data for Top 10 (assuming deg_age & dds_muscle are defined previously)
# Clean row names in counts matrix
clean_ids <- sub("\\.[0-9]+$", "", rownames(counts(dds_muscle)))
rownames_counts_cleaned <- counts(dds_muscle)
rownames(rownames_counts_cleaned) <- clean_ids

# Proceed as before, now subset using compatible rownames
heatmap_data <- rownames_counts_cleaned[rownames(deg_age), ]


# Modify row names to gene symbols
rownames(heatmap_data) <- deg_age$symbol

# Modify column names to include condition info
colnames(heatmap_data) <- paste(colnames(dds_muscle), dds_muscle$condition, sep = "-")

# Filter columns by conditions "ctrl", "age", and "combi"
conditions_to_keep <- c("ctrl", "age", "combi")
cols_to_keep <- grep(paste(conditions_to_keep, collapse="|"), colnames(heatmap_data))
heatmap_data_filtered <- heatmap_data[, cols_to_keep]

# Plot heatmap with adjusted options
options(repr.plot.width=7, repr.plot.height=10)

pheatmap(heatmap_data_filtered,
         #color = colorRampPalette(c("blue","white", "red"))(50),
         cluster_rows = TRUE, cluster_cols = TRUE,
         #show_rownames = TRUE, 
         show_rownames = FALSE, 
         scale = 'row', 
         main = "Heatmap of Aging Genes (Filtered samples)")

In [ ]:
## Rescued-t-normal

In [ ]:
# 17. Generate heatmap of rescued genes expression across filtered samples
# Filter samples to keep only "ctrl", "age", and "combi" conditions
# Filter samples
keep_samples <- colData(dds_muscle)$condition %in% c("ctrl", "age", "combi")

# Extract normalized count matrix
counts_mat <- counts(dds_muscle, normalized = TRUE)

# Clean Ensembl IDs for count matrix rownames
rownames(counts_mat) <- sub("\\.[0-9]+$", "", rownames(counts_mat))

# Subset rescued genes expression and filtered samples
rescued_counts_filtered <- counts_mat[rownames(rescued_to_normal), keep_samples]

# Create annotation dataframe for the filtered samples
sample_anno <- as.data.frame(colData(dds_muscle)[keep_samples, "condition", drop = FALSE])

# Append condition labels to column names for clarity in heatmap
colnames(rescued_counts_filtered) <- paste(colnames(rescued_counts_filtered), sample_anno$condition, sep = "_")

# Ensure rownames of annotation match column names of counts matrix
rownames(sample_anno) <- colnames(rescued_counts_filtered)

# Scale rows (genes) for heatmap visualization (row-wise scaling)
rescued_counts_scaled <- t(scale(t(rescued_counts_filtered)))

# Plot heatmap with pheatmap
library(pheatmap)
pheatmap(rescued_counts_scaled,
         annotation_col = sample_anno,
         main = "Expression Heatmap of Rescued Genes Returned to Normal",
         fontsize_row = 6,
         cluster_cols = FALSE,
         show_rownames = FALSE)


##  GO enrichment plots

In [ ]:
# 14. Visualize GO terms with dotplot and barplot
dotplot(ego_age, showCategory = 20) + ggtitle("GO BP enrichment: Aging DEGs") + theme(plot.title = element_text(hjust = 0.5))
dotplot(ego_rescued, showCategory = 18) + ggtitle("GO BP enrichment: Rescued DEGs") + theme(plot.title = element_text(hjust = 0.5), axis.text.y = element_text(size = 10,  hjust = 1))
dotplot(ego_rescued_to_normal, showCategory = 20) + ggtitle("GO BP enrichmen of rescued-to-normal Genes") + theme(plot.title = element_text(hjust = 0.5))
dotplot(ego_norm_sAct, showCategory=20) + ggtitle("GO Biological Process enrichment of rescued-to-normal driven by sAct genes") + theme(plot.title = element_text(hjust = 0.5))
dotplot(ego_norm_DR, showCategory=18) + ggtitle("GO Biological Process enrichment of rescued-to-normal driven by DR genes") + theme( plot.title = element_text(hjust = 0.5), axis.text.y = element_text(size = 10,  hjust = 1))
dotplot(ego_norm_only_in_combi, showCategory=20) + ggtitle("GO Biological Process enrichment of rescued-to-normal only in combi") + theme(plot.title = element_text(hjust = 0.5))
dotplot(ego_non_rescued, showCategory=20) + ggtitle("GO Biological Process enrichment of genes not rescued by any intervention") + theme(plot.title = element_text(hjust = 0.5))

In [ ]:
# Aging effect GO terms
barplot(ego_age, showCategory=10, title="GO Biological Process enrichment of aging genes") + theme(plot.title = element_text(hjust = 0.5))
# Rescued gene GO terms
barplot(ego_rescued, showCategory=10, title="GO Biological Process enrichment of rescued genes") + theme(plot.title = element_text(hjust = 0.5))

# Rescued to normal genes GO terms
barplot(ego_rescued_to_normal, showCategory=10, title="GO Biological Process enrichment of rescued-to-normal genes") + theme(plot.title = element_text(hjust = 0.5))
barplot(ego_norm_sAct, showCategory=10, title="GO Biological Process enrichment of rescued-to-normal driven by sAct genes") + theme(plot.title = element_text(hjust = 0.5))
barplot(ego_norm_DR, showCategory=10, title="GO Biological Process enrichment of rescued-to-normal driven by DR genes") + theme(plot.title = element_text(hjust = 0.5))
barplot(ego_norm_only_in_combi, showCategory=10, title = "GO Biological Process enrichment of rescued-to-normal only in combi") + theme(plot.title = element_text(hjust = 0.5))
barplot(ego_non_rescued, showCategory=10, title = "GO Biological Process enrichment of genes not rescued by any intervention") + theme(plot.title = element_text(hjust = 0.5))

## GSEA Enrichment Plots

In [ ]:
# Set rownames and remove pathway column
#rownames(nes_mat) <- nes_mat$pathway
#nes_mat_plot <- nes_mat %>% select(-pathway)

# Make sure it’s a numeric matrix
nes_mat_plot <- as.matrix(nes_mat)
nes_mat_plot <- nes_mat %>%
  filter(!is.na(NES_age) & !is.na(NES_intv) & !is.na(NES_rescued)) %>%
  column_to_rownames(var = "pathway") %>%
  as.matrix()


options(repr.plot.width = 12, repr.plot.height = 20)  # inches
p <- pheatmap(nes_mat_plot,
              cluster_rows = TRUE,
              cluster_cols = FALSE,
              main = "Pathway NES, muscle: Aging vs Combined Intervention",
              fontsize_row = 10,
              fontsize_col = 12,
              cellwidth = 25,
              cellheight = 15)

# Overall numbers plot

In [ ]:
# Count total aging DEGs
n_deg_age <- nrow(deg_age)

# Count rescued genes (reversal upon combined intervention)
n_rescued <- nrow(rescued_genes)

# Count genes exclusively rescued by DR XOR sAct (restored > 0 in only one intervention)
nb_exclusive_DR <- effect_comparison %>%
  filter((Restoration_DR > 0 & Restoration_sAct == 0)) %>%
  nrow()

nb_exclusive_sAct <- effect_comparison %>%
  filter((Restoration_sAct > 0 & Restoration_DR == 0)) %>%
  nrow()

# Count genes rescued only in combined intervention (not rescued by DR or sAct alone)
only_combi <- effect_comparison %>%
  filter((Restoration_sAct == 0 & Restoration_DR == 0)) %>%
  nrow()

# Prepare summary data frame for plotting
summary_df <- tibble(
  Category = c("Aging DEGs", "Rescued Genes", "Exclusive DR Rescue", "Exclusive sAct Rescue", "Only Combined Rescue"),
  Count = c(n_deg_age, n_rescued, nb_exclusive_DR, nb_exclusive_sAct, only_combi)
)

# Plot bar chart
ggplot(summary_df, aes(x = Category, y = Count, fill = Category)) +
  geom_bar(stat = "identity") +
  geom_text(aes(label = Count), vjust = -0.5, size = 5) +
  theme_classic() +
  theme(axis.text.x = element_text(angle = 90, hjust = 1)) +
  labs(title = "Muscle Comparison of Differentially Expressed and Rescued Genes",
       y = "Number of Genes",
       x = "")


# Write out bulk muscle DEGs results

In [ ]:
# Write full differential expression results with gene symbols and annotations
write.csv(res_age_df, file = "/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/results_tables/muscle_res_age_results_with_symbols.csv", row.names = TRUE)

# Write out aged genes rescued by non of the interventions
write.csv(non_rescued_df, file = "/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/results_tables/muscle_non_rescued_aged_DEGs.csv", row.names = TRUE)

In [ ]:
# Convert res_intv results to a data frame for merging
res_intv_df <- as.data.frame(res_intv)
rescued_genes$GeneID <- rownames(rescued_genes)
res_intv_df$GeneID <- rownames(res_intv_df)

# Clean Ensembl IDs (remove version suffix) before merging and mapping
rescued_genes$CleanGeneID <- sub("\\.[0-9]+$", "", rescued_genes$GeneID)
res_intv_df$CleanGeneID <- sub("\\.[0-9]+$", "", res_intv_df$GeneID)

# Merge deg_age (with cleaned IDs) and res_intv_df by CleanGeneID for rescued genes
combined_df <- merge(
  rescued_genes[, c("CleanGeneID", "log2FoldChange", "padj")],
  res_intv_df[, c("CleanGeneID", "log2FoldChange", "padj")],
  by = "CleanGeneID",
  suffixes = c("_aging", "_intervention")
)

# Filter merged table to rescued genes only (optional, but they should already align)
rescued_gene_ids <- sub("\\.[0-9]+$", "", rownames(rescued_genes))
combined_rescued_df <- combined_df[combined_df$CleanGeneID %in% rescued_gene_ids, ]

# Map gene symbols for clarity
gene_symbols <- mapIds(org.Mm.eg.db,
                       keys = combined_rescued_df$CleanGeneID,
                       column = "SYMBOL",
                       keytype = "ENSEMBL",
                       multiVals = "first")

combined_rescued_df$Symbol <- gene_symbols
combined_rescued_df$LFC_Sum <- combined_rescued_df$log2FoldChange_aging + combined_rescued_df$log2FoldChange_intervention

# Add logical/rescue significance column (padj_intervention < 0.05)
combined_rescued_df$rescue_significant <- ifelse(combined_rescued_df$padj_intervention < 0.05, "Yes", "No")

# Write merged rescued genes DE info for muscle dataset
write.csv(combined_rescued_df,
          "./aging_bulk_muscle/results_tables/muscle_combined_rescued_genes_DE_info.csv",
          row.names = FALSE)

# Write rescued genes and rescued-to-normal genes with symbols and annotations
write.csv(rescued_genes,
          file = "./aging_bulk_muscle/results_tables/muscle_rescued_genes.csv",
          row.names = TRUE)

write.csv(rescued_to_normal,
          file = "./aging_bulk_muscle/results_tables/muscle_rescued_to_normal.csv",
          row.names = TRUE)

In [ ]:
# --- Export enrichment results ---
write.csv(as.data.frame(ego_age), file = "/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/results_tables/enrichment/muscle_enrichment_results_aging.csv", row.names = FALSE, quote = FALSE)
write.csv(as.data.frame(ego_rescued), file = "/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/results_tables/enrichment/muscle_enrichment_results_rescued.csv", row.names = FALSE, quote = FALSE)

write.csv(as.data.frame(ego_rescued_to_normal), file = "/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/results_tables/enrichment/muscle_enrichment_results_rescued_to_normal.csv", row.names = FALSE, quote = FALSE)
write.csv(as.data.frame(ego_norm_sAct), file = "/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/results_tables/enrichment/muscle_enrichment_results_rescued-to-normal_sAct.csv", row.names = FALSE, quote = FALSE)
write.csv(as.data.frame(ego_norm_DR), file = "/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/results_tables/enrichment/muscle_enrichment_results_rescued-to-normal__DR.csv", row.names = FALSE, quote = FALSE)
write.csv(as.data.frame(ego_non_rescued), file = "/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/results_tables/enrichment/muscle_enrichment_results_non_rescued_genes.csv", row.names = FALSE, quote = FALSE)

# Single-Nucleus RNA-Seq Gene Set Mapping for Muscle

In [ ]:
reticulate::use_condaenv("r-deseq2-env", required = TRUE)
reticulate::py_install("anndata")

## --- LOAD SINGLE NUCLEI DATA FROM H5AD ---

In [ ]:
# 1. Import anndata and read muscle .h5ad file
anndata <- import("anndata")
adata <- anndata$read_h5ad("/data/bonn-epyc/projects/manuela/aging/aging_muscle/annotated-muscle-soupxed2.h5ad")

# 2. Extract count matrix (genes x cells), gene names, and cell metadata
count_matrix <- t(as.matrix(adata$X))
gene_names <- py_to_r(adata$var_names$tolist())
rownames(count_matrix) <- gene_names
cell_metadata <- as.data.frame(py_to_r(adata$obs))
rownames(cell_metadata) <- colnames(count_matrix)

# 3. Create Seurat object
seurat_obj <- CreateSeuratObject(counts = count_matrix, meta.data = cell_metadata)

# 4. Add dimensional reductions if available
if (!is.null(adata$obsm)) {
  pca_mat <- py_to_r(adata$obsm[["X_pca"]])
  umap_mat <- py_to_r(adata$obsm[["X_umap"]])
  rownames(pca_mat) <- colnames(seurat_obj)
  rownames(umap_mat) <- colnames(seurat_obj)
  seurat_obj[["pca"]] <- CreateDimReducObject(embeddings = pca_mat, key = "PC_", assay = DefaultAssay(seurat_obj))
  seurat_obj[["umap"]] <- CreateDimReducObject(embeddings = umap_mat, key = "UMAP_", assay = DefaultAssay(seurat_obj))
}

# 5. Normalize counts manually to create "data" layer
# OLD (throws error):
#counts_mat <- GetAssayData(seurat_obj, assay = "RNA", slot = "counts")

# NEW (Seurat v5):
counts_mat <- LayerData(seurat_obj[["RNA"]], layer = "counts")
total_counts <- Matrix::colSums(counts_mat)
nonzero_cells <- total_counts > 0
norm_counts <- Matrix(0, nrow = nrow(counts_mat), ncol = ncol(counts_mat), dimnames = dimnames(counts_mat))
norm_counts[, nonzero_cells] <- sweep(counts_mat[, nonzero_cells], 2, total_counts[nonzero_cells], "/") * 1e4
log_norm <- log1p(norm_counts)
log_norm[is.na(log_norm)] <- 0
log_norm[is.infinite(log_norm)] <- 0
LayerData(seurat_obj[["RNA"]], layer = "data") <- log_norm
DefaultLayer(seurat_obj[["RNA"]]) <- "data"

### Check Seurat Object & Prepare gene lists

In [ ]:
# 1. Verify your Seurat object assay and features (genes)
DefaultAssay(seurat_obj)  # Should print "RNA" or the assay name you want to use
head(rownames(seurat_obj))  # confirm gene symbols here match those in genes_in_data
features_all <- rownames(seurat_obj)
cat("Number of genes in dataset:", length(features_all), "\n")

In [ ]:
# 2. Prepare your reversal gene list filtered to genes actually present in the Seurat object
reversal_genes <- na.omit(unique(rescued_to_normal$Symbol))
genes_in_data <- reversal_genes[reversal_genes %in% features_all]
cat("Number of reversal genes found in data:", length(genes_in_data), "\n")

if(length(genes_in_data) == 0){
  stop("No reversal genes match gene names in the Seurat object. Please check gene naming conventions.")
}

### Filter for rescued-to-normal (DR, sAct only) genes

In [ ]:
# for reversal_genes
genes_norm <- reversal_genes
genes_norm <- genes_norm[genes_norm %in% rownames(seurat_obj)]
genes_norm <- unique(genes_norm)

# Use exclusive_DR and exclusive_sAct gene symbols from your effect_comparison table
genes_DR <- exclusive_DR$Symbol
genes_DR <- genes_DR[genes_DR %in% rownames(seurat_obj)]  # Ensure genes are in Seurat object
genes_DR <- unique(genes_DR)

genes_sAct <- exclusive_sAct$Symbol
genes_sAct <- genes_sAct[genes_sAct %in% rownames(seurat_obj)]
genes_sAct <- unique(genes_sAct)

genes_aged <- deg_age$Symbol
genes_aged <- genes_aged[genes_aged %in% rownames(seurat_obj)]
genes_aged <- unique(genes_aged)

# Add module scores to Seurat object
seurat_obj <- AddModuleScore(
  object = seurat_obj,
  features = list(genes_norm),
  name = "Reversal_norm"
)

seurat_obj <- AddModuleScore(
  object = seurat_obj,
  features = list(genes_DR),
  name = "Reversal_DR"
)

seurat_obj <- AddModuleScore(
  object = seurat_obj,
  features = list(genes_sAct),
  name = "Reversal_sAct"
)

seurat_obj <- AddModuleScore(
  object = seurat_obj,
  features = list(genes_aged),
  name = "Aged"
)

# Sort scores
# Function to reorder celltype by average expression of a feature
reorder_celltypes <- function(seurat_obj, feature) {
  avg_exp <- FetchData(seurat_obj, vars = c(feature, "celltype")) %>%
    group_by(celltype) %>%
    summarise(mean_exp = mean(get(feature), na.rm = TRUE)) %>%
    arrange(desc(mean_exp))
  factor_levels <- avg_exp$celltype
  seurat_obj$celltype <- factor(seurat_obj$celltype, levels = factor_levels)
  return(seurat_obj)
}

# For Reversal_norm1
seurat_obj <- reorder_celltypes(seurat_obj, "Reversal_norm1")
# To visualize, plot FeaturePlots or ViolinPlots by module score
VlnPlot(seurat_obj, features = "Reversal_norm1", group.by = "celltype", pt.size = 0) +
  ggtitle("Module Score: Reversal back-to-normal Genes") +    # Your custom title here
  theme_classic() +
  theme(
    axis.text.x = element_text(angle = 90, hjust = 1, size = 18),   # Bigger x-axis text
    axis.title.x = element_blank(),
    plot.title = element_text(size = 16),                          # Bigger title
    axis.text.y = element_text(size = 18),                          # Bigger y-axis text
    axis.title.y = element_text(size = 14),
    plot.margin = margin(t = 10, r = 30, b = 10, l = 10)
  )
# For Reversal_DR1
seurat_obj <- reorder_celltypes(seurat_obj, "Reversal_DR1")
# To visualize, plot FeaturePlots or ViolinPlots by module score
VlnPlot(seurat_obj, features = "Reversal_DR1", group.by = "celltype", pt.size = 0) +
  ggtitle("Module Score: Exclusive back-to-normal DR driven Genes") +
  theme_classic() +
  theme(
    axis.text.x = element_text(angle = 90, hjust = 1, size = 18),   # Bigger x-axis text
    axis.title.x = element_blank(),
    plot.title = element_text(size = 16),                          # Bigger title
    axis.text.y = element_text(size = 18),                          # Bigger y-axis text
    axis.title.y = element_text(size = 14),
    plot.margin = margin(t = 10, r = 30, b = 10, l = 10)
  )

# For Reversal_sAct1
seurat_obj <- reorder_celltypes(seurat_obj, "Reversal_sAct1")
# To visualize, plot FeaturePlots or ViolinPlots by module score
VlnPlot(seurat_obj, features = "Reversal_sAct1", group.by = "celltype", pt.size = 0) +
  ggtitle("Module Score: Exclusive back-to-normal sAct driven Genes") +
  theme_classic() +
  theme(
    axis.text.x = element_text(angle = 90, hjust = 1, size = 18),   # Bigger x-axis text
    axis.title.x = element_blank(),
    plot.title = element_text(size = 16),                          # Bigger title
    axis.text.y = element_text(size = 18),                          # Bigger y-axis text
    axis.title.y = element_text(size = 14),
    plot.margin = margin(t = 10, r = 30, b = 10, l = 10)
  )

# For Aged
seurat_obj <- reorder_celltypes(seurat_obj, "Aged1")
# To visualize, plot FeaturePlots or ViolinPlots by module score
VlnPlot(seurat_obj, features = "Aged1", group.by = "celltype", pt.size = 0) +
  ggtitle("Module Score: Aged Genes") +
  theme_classic() +
  theme(
    axis.text.x = element_text(angle = 90, hjust = 1, size = 18),   # Bigger x-axis text
    axis.title.x = element_blank(),
    plot.title = element_text(size = 16),                          # Bigger title
    axis.text.y = element_text(size = 18),                          # Bigger y-axis text
    axis.title.y = element_text(size = 14),
    plot.margin = margin(t = 10, r = 30, b = 10, l = 10)
  )

In [ ]:
#Reversal Score Tables
group_col <- if ("celltype" %in% colnames(seurat_obj@meta.data)) "celltype" else "seurat_clusters"

# Combined reversal score table, sorted descending
avg_score_combined <- seurat_obj@meta.data %>%
  group_by(across(all_of(group_col))) %>%
  summarise(AverageReversalCombined = mean(Reversal_norm1, na.rm = TRUE)) %>%
  arrange(desc(AverageReversalCombined))

# DR reversal score table, sorted descending
avg_score_DR <- seurat_obj@meta.data %>%
  group_by(across(all_of(group_col))) %>%
  summarise(AverageReversalDR = mean(Reversal_DR1, na.rm = TRUE)) %>%
  arrange(desc(AverageReversalDR))

# sAct reversal score table, sorted descending
avg_score_sAct <- seurat_obj@meta.data %>%
  group_by(across(all_of(group_col))) %>%
  summarise(AverageReversalsAct = mean(Reversal_sAct1, na.rm = TRUE)) %>%
  arrange(desc(AverageReversalsAct))


# sAct reversal score table, sorted descending
avg_score_aged <- seurat_obj@meta.data %>%
  group_by(across(all_of(group_col))) %>%
  summarise(AverageAged = mean(Aged1, na.rm = TRUE)) %>%
  arrange(desc(AverageAged))

print(avg_score_combined)
print(avg_score_DR)
print(avg_score_sAct)
print(avg_score_aged)

 ### Save or Export the Results for Downstream Analysis or Visualization

In [ ]:
# Extract metadata columns of interest - replace 'clusters' with your actual clustering column name
metadata_with_reversal <- seurat_obj@meta.data[, c("Reversal_norm1", "Reversal_DR1", "Reversal_sAct1", "celltype")]

# Write to CSV
write.csv(metadata_with_reversal, file = "./muscle_ReversalGeneScores_perCell_seurat.csv", row.names = TRUE)

In [ ]:
colnames(seurat_obj@meta.data)  # search for "aged" or "aging"

# Then add it:
metadata_aging <- seurat_obj@meta.data[, c("Aged1", "celltype")]
write.csv(metadata_aging, "./muscle_AgingGeneScores_perCell_seurat.csv", row.names = TRUE)

## Identify marker genes per cell type and filter for contributing genes

In [ ]:
# Set cell type identities
Idents(seurat_obj) <- seurat_obj$celltype  # or seurat_clusters

# 1. Find markers for all cell types/clusters
markers_all <- FindAllMarkers(seurat_obj, only.pos = TRUE, min.pct = 0.1, logfc.threshold = 0.25)

# Prepare gene lists (cleaned, no NA)
reversal_genes_clean <- reversal_genes[!is.na(reversal_genes) & reversal_genes != ""]
aged_genes_clean <- deg_age$Symbol[!is.na(deg_age$Symbol) & deg_age$Symbol != ""]
aged_genes_only_rescued_in_combi_clean <- only_rescued_in_combi$Symbol[!is.na(only_rescued_in_combi$Symbol) & only_rescued_in_combi$Symbol != ""]

# 2. Subset markers for reversal genes and aged genes
markers_reversal <- markers_all[markers_all$gene %in% reversal_genes_clean, ]
markers_aged <- markers_all[markers_all$gene %in% aged_genes_clean, ]
markers_only_rescued_in_combi <- markers_all[markers_all$gene %in% aged_genes_only_rescued_in_combi_clean, ]

In [ ]:
# 3. Save full markers and subsets
write.csv(markers_all, "/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/results_tables/celltype_markers/muscle_markers_all_celltypes.csv", row.names = FALSE)
write.csv(markers_reversal, "/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/results_tables/celltype_markers/muscle_markers_reversal_genes_celltypes.csv", row.names = FALSE)
write.csv(markers_aged, "/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/results_tables/celltype_markers/muscle_markers_aged_genes_celltypes.csv", row.names = FALSE)
write.csv(markers_only_rescued_in_combi, "/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/results_tables/celltype_markers/muscle_markers_aged_genes_celltypes_only_rescued_in_combi.csv", row.names = FALSE)

# Replace '/' with '_' or other safe character in cluster names
# Function to sanitize cluster names to safe file names
sanitize_name <- function(x) {
  # Replace "/" with "_", you can add other replacements here if necessary
  gsub("/", "_", x)
}

# Use sanitized cluster names for writing files
safe_clusters_reversal <- sanitize_name(unique(markers_reversal$cluster))
for (ct in safe_clusters_reversal) {
  sub_df <- markers_reversal[sanitize_name(markers_reversal$cluster) == ct, ]
  write.csv(sub_df,
            paste0("/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/results_tables/celltype_markers/muscle_markers_reversal_", ct, ".csv"),
            row.names = FALSE)
}

safe_clusters_aged <- sanitize_name(unique(markers_aged$cluster))
for (ct in safe_clusters_aged) {
  sub_df <- markers_aged[sanitize_name(markers_aged$cluster) == ct, ]
  write.csv(sub_df,
            paste0("/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/results_tables/celltype_markers/muscle_markers_aged_", ct, ".csv"),
            row.names = FALSE)
}

# For markers_only_rescued_in_combi, make sure to get unique safely sanitized clusters
safe_clusters_only_rescued_in_combi <- sanitize_name(unique(markers_only_rescued_in_combi$cluster))
for (ct in safe_clusters_only_rescued_in_combi) {
  sub_df <- markers_only_rescued_in_combi[sanitize_name(markers_only_rescued_in_combi$cluster) == ct, ]
  write.csv(sub_df,
            paste0("/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/results_tables/celltype_markers/muscle_markers_only_rescued_in_combi_", ct, ".csv"),
            row.names = FALSE)
}

### Visualization % of genes per cell type 

In [ ]:
# Sanitize cluster names again if needed
markers_only_rescued_in_combi$cluster_safe <- gsub("/", "_", markers_only_rescued_in_combi$cluster)

# Count distinct genes per cluster
gene_count_table <- markers_only_rescued_in_combi %>%
  group_by(cluster_safe) %>%
  summarise(gene_count = n_distinct(gene))  # Replace 'gene' with your gene column name

# Print the table
print(gene_count_table)

# Optionally, write to CSV
write.csv(gene_count_table, "/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/results_tables/celltype_markers/muscle_gene_counts_only_rescued_in_combi.csv",
          row.names = FALSE)

### GOBP enrichment per cell type (for aged and rescued markers)

In [ ]:
markers_aged <- read.csv("/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/results_tables/celltype_markers/muscle_markers_aged_genes_celltypes.csv")
markers_rescued <- read.csv("/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/results_tables/celltype_markers/muscle_markers_reversal_genes_celltypes.csv")

In [ ]:
# Define your marker tables and analysis groups
marker_tables <- list(
  aged = markers_aged,
  rescued = markers_rescued
)

# Where to save enrichment tables and plots
table_dir <- "/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/results_tables/celltype_pathways/"
plot_dir <- file.path(table_dir, "enrichment_plots")

if (!dir.exists(table_dir)) dir.create(table_dir, recursive=TRUE)
if (!dir.exists(plot_dir)) dir.create(plot_dir, recursive=TRUE)

# Function to replace problematic characters in filenames
make_safe <- function(name) gsub("[/\\\\]", "_", name)

for (groupname in names(marker_tables)) {
  markers_df <- marker_tables[[groupname]]
  celltypes <- unique(markers_df$cluster)
  pathway_results <- list()
  
  # Run pathway analysis per cell type
  for (ct in celltypes) {
    genes <- markers_df$gene[markers_df$cluster == ct]
    entrez_ids <- na.omit(mapIds(org.Mm.eg.db, keys=genes, column="ENTREZID", keytype="SYMBOL", multiVals="first"))
    
    ego <- enrichGO(
      gene = entrez_ids,
      OrgDb = org.Mm.eg.db,
      keyType = "ENTREZID",
      ont = "BP",
      pAdjustMethod = "BH",
      pvalueCutoff = 0.05,
      qvalueCutoff = 0.2,
      readable = TRUE
    )
    pathway_results[[ct]] <- ego
    
    # Save enrichment table (as .csv)
    safe_ct <- make_safe(ct)
    if (!is.null(ego) & nrow(as.data.frame(ego)) > 0) {
      write.csv(as.data.frame(ego),
        file = file.path(table_dir, sprintf("gobp_%s_%s.csv", groupname, safe_ct)),
        row.names = FALSE
      )
    }
    
    # Plot and save barplot/dotplot
    if (!is.null(ego) & nrow(as.data.frame(ego)) > 0) {
      png(file = file.path(plot_dir, sprintf("GO_BP_barplot_%s_%s.png", groupname, safe_ct)), width=1000, height=800)
      print(barplot(ego, showCategory=10, title=paste("GO BP Enrichment -", groupname, ct)))
      dev.off()
      
      png(file = file.path(plot_dir, sprintf("GO_BP_dotplot_%s_%s.png", groupname, safe_ct)), width=1000, height=800)
      print(dotplot(ego, showCategory=10, title=paste("GO BP Enrichment -", groupname, ct)))
      dev.off()
    }
  }
}

In [ ]:
output_dir <- "/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/results_tables/celltype_pathways/"

if (!dir.exists(output_dir)) {
  dir.create(output_dir, recursive = TRUE)
}

# Sanitize cell type names for safe filenames
safe_names <- gsub("[/\\\\]", "_", names(pathway_results))  # Replace / or \ with _

for (i in seq_along(pathway_results)) {
  ct <- names(pathway_results)[i]
  safe_ct <- safe_names[i]
  
  if (!is.null(pathway_results[[ct]])) {
    df <- as.data.frame(pathway_results[[ct]])
    write.csv(df, file.path(output_dir, paste0("gobp_", safe_ct, ".csv")), row.names = FALSE)
  }
}

In [ ]:
output_dir <- "/data/bonn-epyc/projects/manuela/aging/aging_bulk_muscle/results_tables/celltype_pathways/enrichment_plots/"

if (!dir.exists(output_dir)) dir.create(output_dir, recursive = TRUE)

safe_names <- gsub("[/\\\\]", "_", names(pathway_results))

for (i in seq_along(pathway_results)) {
  ct <- names(pathway_results)[i]
  safe_ct <- safe_names[i]
  ego <- pathway_results[[ct]]
  
  if (!is.null(ego)) {
    png(filename = file.path(output_dir, paste0("GO_BP_barplot_", safe_ct, ".png")), width = 1000, height = 800)
    print(barplot(ego, showCategory=10, title=paste("GO BP Enrichment -", ct)))
    dev.off()

    png(filename = file.path(output_dir, paste0("GO_BP_dotplot_", safe_ct, ".png")), width = 1000, height = 800)
    print(dotplot(ego, showCategory=10, title=paste("GO BP Enrichment -", ct)))
    dev.off()
  }
}

# Cell–Cell Communication Analysis LR based (CellChat)

## use previously generated objects

In [ ]:
# Load your saved CellChat objects with centrality computed:
cellchat_rgzj1 <- readRDS("/data/bonn-epyc/projects/manuela/aging/aging_muscle/cellchat_objects/cellchat_rgzj1_centr.rds") #ctrl
cellchat_rgzj2 <- readRDS("/data/bonn-epyc/projects/manuela/aging/aging_muscle/cellchat_objects/cellchat_rgzj2_centr.rds") #age
cellchat_rgzj3 <- readRDS("/data/bonn-epyc/projects/manuela/aging/aging_muscle/cellchat_objects/cellchat_rgzj3_centr.rds") #sAct
cellchat_rgzj4 <- readRDS("/data/bonn-epyc/projects/manuela/aging/aging_muscle/cellchat_objects/cellchat_rgzj4_centr.rds") #DR
cellchat_rgzj5 <- readRDS("/data/bonn-epyc/projects/manuela/aging/aging_muscle/cellchat_objects/cellchat_rgzj5_centr.rds") #combi


# Your reversal genes list (make sure symbols exactly match CellChat gene symbols)
reversal_genes <- na.omit(unique(rescued_to_normal$Symbol))

# 1. Extract all inferred ligand-receptor interactions from CellChat object as data frame
# Default slot.name = "net" returns cell-cell communication interactions at ligand-receptor pair level
all_communications_rgzj1 <- subsetCommunication(cellchat_rgzj1, slot.name = "net")

# 2. Filter rows where ligand or receptor is in reversal genes
lr_filtered <- all_communications_rgzj1 %>%
  filter(ligand %in% reversal_genes | receptor %in% reversal_genes)

cat("Number of ligand-receptor interactions involving reversal genes:", nrow(lr_filtered), "\n")
head(lr_filtered)

# 3. Identify signaling pathways that include these reversal genes as ligand or receptor
all_pathways <- cellchat_rgzj1@netP$pathways

# Helper function to check gene involvement
pathway_has_reversal_genes <- function(pathway) {
  pairs_df <- cellchat_rgzj1@LR$pairs[[pathway]]
  any(pairs_df$ligand %in% reversal_genes) || any(pairs_df$receptor %in% reversal_genes)
}

pathways_of_interest <- Filter(pathway_has_reversal_genes, all_pathways)
cat("Signaling pathways with reversal genes:\n")
print(pathways_of_interest)

# 4. Visualize communication networks for these pathways
if(length(pathways_of_interest) > 0) {
  for (p in pathways_of_interest) {
    netVisual_circle(cellchat_rgzj1@net[[p]]$count,
                     vertex.weight = table(cellchat_rgzj1@idents),
                     weight.scale = TRUE,
                     title.name = paste("Cell-cell communication network:", p))
  }
} else {
  cat("No signaling pathways involving reversal genes found.\n")
}

# 5. Optionally save filtered ligand-receptor interactions
write.csv(lr_filtered, file = "./cellchat_outputs_muscle/rgzj1_lr_interactions_reversal_genes.csv", row.names = FALSE)

In [ ]:


# reversal genes as symbols
reversal_genes <- na.omit(unique(rescued_to_normal$Symbol))

analyze_reversal_cellchat <- function(obj, name, rev_genes) {
  all_comm <- subsetCommunication(obj, slot.name = "net")
  lr_filt <- all_comm %>%
    filter(ligand %in% rev_genes | receptor %in% rev_genes)
  cat(name, ": LR with reversal genes:", nrow(lr_filt), "\n")
  
  # Extract pathways directly from filtered LR table column 'pathway_name' (like your working code)
  pathways_of_interest <- unique(lr_filt$pathway_name)
  cat(name, ": pathways with reversal genes:\n")
  print(pathways_of_interest)
  
  if (length(pathways_of_interest) > 0) {
    dir.create("cellchat_outputs_muscle", showWarnings = FALSE)
    
    for (p in pathways_of_interest) {
      if (p %in% dimnames(obj@netP$prob)[[3]]) {
        png(file = paste0("cellchat_outputs_muscle/", name, "_", gsub("[^A-Za-z0-9]", "_", p), ".png"), 
            width = 800, height = 800, res = 100)
        
        netVisual_circle(obj@netP$prob[,,p],
                         vertex.weight = table(obj@idents),
                         weight.scale = TRUE,
                         title.name = paste(name, ":", p))
        
        dev.off()
        cat("Saved plot:", paste0(name, "_", gsub("[^A-Za-z0-9]", "_", p), ".png"), "\n")
      } else {
        cat("Warning: Pathway", p, "not found in netP probabilities for", name, "\n")
      }
    }
  } else {
    cat(name, ": no signaling pathways involving reversal genes found.\n")
  }
  
  write.csv(lr_filt,
            file = paste0("cellchat_outputs_muscle/", name, "_lr_interactions_reversal_genes.csv"),
            row.names = FALSE)
}

# Create named list of your CellChat objects
cellchat_list <- list(
  ctrl = cellchat_rgzj1,
  age = cellchat_rgzj2,
  sAct = cellchat_rgzj3,
  DR = cellchat_rgzj4,
  combi = cellchat_rgzj5
)

# Run the analysis and save plots for all samples
for (nm in names(cellchat_list)) {
  analyze_reversal_cellchat(cellchat_list[[nm]], nm, reversal_genes)
}

# Write out files for additional plots:

In [ ]:
# Export all module scores in one go
df <- data.frame(
  celltype        = seurat_obj$celltype,
  Reversal_norm   = seurat_obj@meta.data$Reversal_norm1,
  Reversal_DR     = seurat_obj@meta.data$Reversal_DR1,
  Reversal_sAct   = seurat_obj@meta.data$Reversal_sAct1,
  Aged            = seurat_obj@meta.data$Aged1
)
write.csv(df, "muscle_module_scores_all.csv", row.names = FALSE)

cat("Exported", nrow(df), "cells,", length(unique(df$celltype)), "cell types\n")
cat("Columns:", colnames(df), "\n")

In [ ]:
colnames(seurat_obj@meta.data)

In [ ]:
table(seurat_obj$sample)

In [ ]:
# Muscle (same):
df <- data.frame(
  celltype  = seurat_obj$celltype,
  condition = seurat_obj$sample
)
write.csv(df, "muscle_celltype_condition.csv", row.names = FALSE)

In [ ]:
# Muscle (same)
# In R, using the anndata object that's already loaded:
umap_coords <- py_to_r(adata$obsm[["X_umap"]])
cell_metadata <- as.data.frame(py_to_r(adata$obs))

df <- data.frame(
  celltype  = cell_metadata$celltype,
  condition = cell_metadata$sample,   # or however condition is stored in the h5ad
  UMAP_1    = umap_coords[, 1],
  UMAP_2    = umap_coords[, 2]
)
write.csv(df, "muscle_umap_coordinates.csv", row.names = FALSE)

# Check Gene Sets of interest

In [ ]:
# List of muscle key genes to validate patterns (down in Ercc1, rescued by intervention)
muscle_genes <- c("Ryr3", "Kcnq5", "Nav3", "Ptprm", "Wwox", "Oxr1", "Nrros", "Gpx1", "Ttn", "Tgfb1")

# Map symbol to Ensembl IDs if needed
muscle_ensembl <- mapIds(org.Mm.eg.db, keys = muscle_genes, column = "ENSEMBL", keytype = "SYMBOL", multiVals = "first")

# Assuming you have muscle effect_comparison or equivalent DE results
muscle_subset <- effect_comparison[effect_comparison$Symbol %in% muscle_genes, ]

# Print and inspect fold changes across conditions (age, DR, sAct, combined)
print(muscle_subset[, c("Symbol", "LFC_Age", "LFC_DR", "LFC_sAct", "LFC_Combined")])

# Plot visualization as above (optional)
df_long_muscle <- melt(muscle_subset[, c("Symbol", "LFC_Age", "LFC_DR", "LFC_sAct", "LFC_Combined")], id.vars = "Symbol")
ggplot(df_long_muscle, aes(x = variable, y = value, fill = variable)) +
  geom_bar(stat="identity", position="dodge") +
  facet_wrap(~Symbol, scales = "free_y") +
  theme_bw() +
  labs(title = "Expression Changes of Key Muscle Genes Across Conditions", y = "Log2 Fold Change", x = "Condition")

# Write out packages

In [ ]:
sessionInfo()

In [ ]:
pkgs <- installed.packages(fields = c("Package", "Version"))[, c("Package", "Version")]
write.csv(pkgs, file = "/data/bonn-epyc/projects/manuela/aging/installed_packages_muscle.csv", row.names = FALSE)